# InfoGain 分析

分析模型推理链（Thought + Action）对预测 tool_call 和 Action 的影响。

## 分析目标

1. **推理链价值评估**: 统计不同推理链对 tool_call 和 Action 的帮助
2. **模型 vs GT CoT 对比**: 评估模型生成的 CoT 质量
3. **InfoGain 作为 RL Reward**: 探讨用 InfoGain 筛选 CoT 的可行性

In [ ]:
import json\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom pathlib import Path\nfrom typing import Dict, List, Tuple\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# 设置中文字体\nplt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']\nplt.rcParams['axes.unicode_minus'] = False\n\n# 设置显示选项\npd.set_option('display.max_columns', None)\npd.set_option('display.max_rows', 100)\npd.set_option('display.width', 1000)

## 1. 数据加载和预处理

In [ ]:
# 读取 InfoGain 结果
result_file = Path("../guir1/outputs/Qwen3-VL-4B-Instruct/analysis/agentnetbench_test_qwen3vl_sampling16_Thought:_infogain_multiGPU.json")

with open(result_file, 'r') as f:
    data = json.load(f)

print(f"总查询数: {data['summary']['total queries']}")
print(f"总样本数: {data['summary']['total_samples']}")
print(f"详细结果数: {len(data['detailed_results'])}")

In [ ]:
def fill_gt_perplexities(sample_infogains: List[Dict]) -> List[Dict]:
    """填充 GT 相关的 perplexity 值
    
    Ground truth 相关的对照组只在 sample_idx=0 时计算，
    需要将这些值复制到其他样本中。
    
    正确处理 inf, -inf, None, NaN 值。
    """
    import math
    
    def is_invalid_value(value):
        """检查值是否无效（inf, -inf, None, NaN）"""
        if value is None:
            return True
        if isinstance(value, float):
            if math.isinf(value) or math.isnan(value):
                return True
        return False
    
    # 找到 sample_idx=0 的 GT 值
    gt_values = None
    for sample in sample_infogains:
        if sample['sample_id'] == 0:
            gt_values = {
                'perplexity_gt_action_toolcall': sample.get('perplexity_gt_action_toolcall'),
                'perplexity_gt_thought_action_toolcall': sample.get('perplexity_gt_thought_action_toolcall'),
                'perplexity_gt_full_toolcall': sample.get('perplexity_gt_full_toolcall'),
                'perplexity_baseline_action': sample.get('perplexity_baseline_action'),
                'perplexity_gt_thought_action': sample.get('perplexity_gt_thought_action'),
                'perplexity_gt_full_action': sample.get('perplexity_gt_full_action'),
                'gt_action_infogain': sample.get('gt_action_infogain'),
                'gt_thought_action_infogain': sample.get('gt_thought_action_infogain'),
                'gt_full_infogain': sample.get('gt_full_infogain'),
                'gt_thought_to_action_infogain': sample.get('gt_thought_to_action_infogain'),
                'gt_full_to_action_infogain': sample.get('gt_full_to_action_infogain'),
            }
            break
    
    if gt_values is None:
        print("警告: 未找到 sample_idx=0 的 GT 值")
        return sample_infogains
    
    # 验证 sample_id=0 的值是否有效
    invalid_keys = [k for k, v in gt_values.items() if is_invalid_value(v)]
    if invalid_keys:
        print(f"警告: sample_id=0 中以下 GT 值无效: {invalid_keys}")
    
    # 填充其他样本的 GT 值
    filled_samples = []
    for sample in sample_infogains:
        sample_copy = sample.copy()
        for key, value in gt_values.items():
            current_value = sample_copy.get(key)
            if is_invalid_value(current_value):
                sample_copy[key] = value
        filled_samples.append(sample_copy)
    
    return filled_samples

# 填充所有 query 的 GT 值
for query_result in data['detailed_results']:
    query_result['sample_infogains'] = fill_gt_perplexities(query_result['sample_infogains'])

print("✓ GT perplexity 值已填充")

In [ ]:
# 将数据转换为 DataFrame 以便分析\nrows = []\nfor query_result in data['detailed_results']:\n    query_id = query_result['query_id']\n    for sample in query_result['sample_infogains']:\n        row = {'query_id': query_id}\n        row.update(sample)\n        rows.append(row)\n\ndf = pd.DataFrame(rows)\n\n# 过滤掉无效样本（perplexity 为 inf 或 None）\ndf = df[df['perplexity_baseline_toolcall'] != float('inf')]\ndf = df[df['perplexity_baseline_toolcall'].notna()]\n\nprint(f\"有效样本数: {len(df)}\")\nprint(f\"\\n数据列:\")\nprint(df.columns.tolist())

In [ ]:
# 查看数据摘要\nprint(\"=\" * 80)\nprint(\"InfoGain 统计摘要\")\nprint(\"=\" * 80)\n\ninfogain_cols = [\n    'model_action_infogain',\n    'model_thought_action_infogain',\n    'gt_action_infogain',\n    'gt_thought_action_infogain',\n    'gt_full_infogain',\n    'model_thought_to_action_infogain',\n    'gt_thought_to_action_infogain',\n    'gt_full_to_action_infogain',\n]\n\ndf[infogain_cols].describe()

## 2. 推理链价值分析\n\n### 2.1 对于 tool_call 的推理链价值

In [ ]:
def classify_toolcall_benefit(row) -> str:\n    \"\"\"根据不同推理链的 InfoGain 分类样本受益类型\"\"\"\n    threshold = 0.05\n    \n    model_action_benefit = row['model_action_infogain'] > threshold\n    model_thought_action_benefit = row['model_thought_action_infogain'] > threshold\n    gt_action_benefit = row['gt_action_infogain'] > threshold\n    gt_thought_action_benefit = row['gt_thought_action_infogain'] > threshold\n    gt_full_benefit = row['gt_full_infogain'] > threshold\n    \n    benefits = [model_action_benefit, model_thought_action_benefit, gt_action_benefit, gt_thought_action_benefit, gt_full_benefit]\n    benefit_count = sum(benefits)\n    \n    if benefit_count == 0:\n        return \"无增益\"\n    elif benefit_count == 1:\n        if model_action_benefit:\n            return \"仅模型Action有益\"\n        elif model_thought_action_benefit:\n            return \"仅模型Thought+Action有益\"\n        elif gt_action_benefit:\n            return \"仅GT Action有益\"\n        elif gt_thought_action_benefit:\n            return \"仅GT Thought+Action有益\"\n        else:\n            return \"仅GT Full有益\"\n    else:\n        return \"混合受益\"\n\ndf['toolcall_benefit_type'] = df.apply(classify_toolcall_benefit, axis=1)\n\n# 统计各类型数量\nbenefit_counts = df['toolcall_benefit_type'].value_counts()\nbenefit_percentages = df['toolcall_benefit_type'].value_counts(normalize=True) * 100\n\nprint(\"=\" * 80)\nprint(\"对于 tool_call 的推理链受益类型统计\")\nprint(\"=\" * 80)\nfor benefit_type in benefit_counts.index:\n    count = benefit_counts[benefit_type]\n    pct = benefit_percentages[benefit_type]\n    print(f\"{benefit_type:30s}: {count:4d} ({pct:5.2f}%)\")

In [ ]:
# 可视化：tool_call 推理链受益类型分布\nfig, ax = plt.subplots(figsize=(12, 6))\nbenefit_counts.plot(kind='bar', ax=ax, color='steelblue')\nax.set_title('对于 tool_call 的推理链受益类型分布', fontsize=14, fontweight='bold')\nax.set_xlabel('受益类型', fontsize=12)\nax.set_ylabel('样本数量', fontsize=12)\nax.grid(axis='y', alpha=0.3)\nplt.xticks(rotation=45, ha='right')\n\n# 添加百分比标注\nfor i, (count, pct) in enumerate(zip(benefit_counts, benefit_percentages)):\n    ax.text(i, count + 0.5, f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)\n\nplt.tight_layout()\nplt.show()